Phase 6 (stretch) — Two-Tower retrieval model (DSSM-style). DeepFM is the precise ranker but scoring all 9,022 candidates per request is too slow for real-time push (S1). Two-Tower trains a user tower and an item tower that each project to the same 32-dim embedding space; at serve time all item embeddings are pre-computed once, and a user's request only needs one tower forward + a dot-product against the precomputed matrix to get the top-200 candidates. DeepFM then only ranks those 200, cutting serving latency from seconds to tens of milliseconds. Goal here is to confirm Recall@200 ≥ 90% (so the candidate set still contains the true positive most of the time) and measure the latency win.

In [ ]:
import json
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import platform

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s",
                    datefmt="%H:%M:%S", force=True)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"arch: {platform.machine()}  device: {DEVICE}")

Inline IDEncoder + dataset + eval (same shape as 05a, copied for self-containment).

In [ ]:
class IDEncoder:
    def __init__(self, ids, oov_token="<UNK>"):
        oov_markers = {"<NEW_USER>", "<NEW_BUSINESS>", "<UNK>", oov_token}
        unique_real_ids = sorted({i for i in ids if i not in oov_markers})
        self.id_to_idx = {oov_token: 0}
        for idx, _id in enumerate(unique_real_ids, start=1):
            self.id_to_idx[_id] = idx
        for marker in oov_markers:
            self.id_to_idx.setdefault(marker, 0)
        self._size = 1 + len(unique_real_ids)

    def __len__(self):
        return self._size

    def encode(self, _id):
        return self.id_to_idx.get(_id, 0)

    def encode_array(self, ids):
        return ids.map(self.id_to_idx).fillna(0).astype(np.int64).values


class TasteHunterDataset(Dataset):
    def __init__(self, df, user_encoder, item_encoder, user_features, item_features):
        self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))
        self.item_idx = torch.from_numpy(item_encoder.encode_array(df["business_id"]))
        self.label = torch.from_numpy(df["label"].astype(np.float32).values)
        n_users = len(user_encoder)
        n_items = len(item_encoder)
        self.user_num = np.zeros((n_users, 6), dtype=np.float32)
        self.user_cuisine = np.zeros((n_users, 50), dtype=np.float32)
        for _, row in user_features.iterrows():
            uidx = user_encoder.encode(row["user_id"])
            self.user_num[uidx] = [row["avg_rating_given"], row["review_count_log"], row["days_active"],
                                    row["elite_flag"], row["mean_distance_traveled"], row["price_tolerance_avg"]]
            emb = row["fav_cuisine_emb"]
            if isinstance(emb, list) and len(emb) == 50:
                self.user_cuisine[uidx] = emb
        self.item_num = np.zeros((n_items, 7), dtype=np.float32)
        self.item_cat = np.zeros((n_items, 50), dtype=np.float32)
        for _, row in item_features.iterrows():
            iidx = item_encoder.encode(row["business_id"])
            self.item_num[iidx] = [row["avg_rating"], row["review_count_log"], row["price_level"],
                                    row["is_open"], row["has_outdoor_seating"], row["photo_count_log"], row["city_id"]]
            cat = row["categories_multi_hot"]
            if isinstance(cat, list) and len(cat) == 50:
                self.item_cat[iidx] = cat
        self.user_num_t = torch.from_numpy(self.user_num)
        self.user_cuisine_t = torch.from_numpy(self.user_cuisine)
        self.item_num_t = torch.from_numpy(self.item_num)
        self.item_cat_t = torch.from_numpy(self.item_cat)

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        u = self.user_idx[idx]
        i = self.item_idx[idx]
        return {
            "user_idx": u, "item_idx": i, "label": self.label[idx],
            "user_num": self.user_num_t[u], "user_cuisine": self.user_cuisine_t[u],
            "item_num": self.item_num_t[i], "item_cat": self.item_cat_t[i],
        }


def make_val_eval_pairs(val_df, user_encoder, item_encoder, item_features, n_negs=99, rng_seed=42):
    """Same 1+99 sampled eval used for DeepFM, lets us compare ranking metrics consistently."""
    rng = np.random.default_rng(rng_seed)
    val_pos = val_df[val_df["stars"] >= 4].copy()
    biz_city = item_features.set_index("business_id")["city"].to_dict()
    val_pos["city"] = val_pos["business_id"].map(biz_city).fillna("<UNK>")
    city_biz = {}
    for bid, c in biz_city.items():
        if c == "<UNK>":
            continue
        city_biz.setdefault(c, []).append(bid)
    for c in city_biz:
        city_biz[c] = np.array(city_biz[c])
    all_user, all_item, all_label = [], [], []
    for row in val_pos.itertuples(index=False):
        if row.city not in city_biz:
            continue
        candidates = city_biz[row.city]
        sampled = rng.choice(candidates, size=n_negs * 2, replace=True)
        negs = [b for b in sampled if b != row.business_id][:n_negs]
        if len(negs) < n_negs:
            continue
        items = [row.business_id] + negs
        all_user.append([user_encoder.encode(row.user_id)] * (n_negs + 1))
        all_item.append([item_encoder.encode(b) for b in items])
        all_label.append([1.0] + [0.0] * n_negs)
    return (np.array(all_user, dtype=np.int64),
            np.array(all_item, dtype=np.int64),
            np.array(all_label, dtype=np.float32))

**Two-Tower model.** User tower and item tower are two independent MLPs with no shared weights. Both project their inputs into the same 32-dim space, and the score is just the dot product. Independent towers are the whole point — at serve time you encode all items once, dump them as a (n_items, 32) matrix, and a user query only needs the user-tower forward + a single matrix-vector product.

In [ ]:
class UserTower(nn.Module):
    def __init__(self, n_users, emb_dim=8, num_dim=6, cuisine_dim=50, hidden=(128, 64), out_dim=32, dropout=0.1):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        layers = []
        prev = emb_dim + num_dim + cuisine_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_idx, user_num, user_cuisine):
        u = self.user_emb(user_idx)
        x = torch.cat([u, user_num, user_cuisine], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)


class ItemTower(nn.Module):
    def __init__(self, n_items, emb_dim=8, num_dim=7, cat_dim=50, hidden=(128, 64), out_dim=32, dropout=0.1):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        layers = []
        prev = emb_dim + num_dim + cat_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, item_idx, item_num, item_cat):
        i = self.item_emb(item_idx)
        x = torch.cat([i, item_num, item_cat], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, out_dim=32, temperature=10.0):
        super().__init__()
        self.user_tower = UserTower(n_users, out_dim=out_dim)
        self.item_tower = ItemTower(n_items, out_dim=out_dim)
        self.temperature = temperature

    def forward(self, user_idx, item_idx, user_num, user_cuisine, item_num, item_cat):
        u = self.user_tower(user_idx, user_num, user_cuisine)
        i = self.item_tower(item_idx, item_num, item_cat)
        score = (u * i).sum(dim=-1) * self.temperature
        return torch.sigmoid(score)

Load data, build encoders, dataset, eval pairs.

In [ ]:
user_features = pd.read_parquet(FEATURES_DIR / "user_features.parquet")
item_features = pd.read_parquet(FEATURES_DIR / "item_features.parquet")
val_df = pd.read_parquet(CLEANED_DIR / "val_reviews.parquet")
train_df = pd.read_parquet(FEATURES_DIR / "train_with_negatives.parquet")

user_encoder = IDEncoder(user_features["user_id"].tolist(), oov_token="<NEW_USER>")
item_encoder = IDEncoder(item_features["business_id"].tolist(), oov_token="<NEW_BUSINESS>")
n_users, n_items = len(user_encoder), len(item_encoder)

user_idx_eval, item_idx_eval, label_eval = make_val_eval_pairs(
    val_df, user_encoder, item_encoder, item_features, n_negs=99,
)
print(f"val eval: {user_idx_eval.shape[0]} users x {user_idx_eval.shape[1]} candidates")

t0 = time.time()
train_ds = TasteHunterDataset(train_df, user_encoder, item_encoder, user_features, item_features)
print(f"train dataset: {len(train_ds)} samples ({time.time()-t0:.1f}s)")
train_loader = DataLoader(train_ds, batch_size=8192, shuffle=True, num_workers=0, pin_memory=True)

Train Two-Tower with the same 1:4 negatives and BCE loss as DeepFM. The temperature scaling on the dot product is a standard DSSM trick — without it, the cosine values are bounded in [-1, 1] and BCE saturates.

In [ ]:
CONFIG = {"out_dim": 32, "temperature": 10.0, "lr": 1e-3, "l2": 1e-5, "epochs": 10}
print(f"config: {CONFIG}")

model = TwoTower(n_users, n_items, out_dim=CONFIG["out_dim"], temperature=CONFIG["temperature"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"])
bce = nn.BCELoss()

history = {"epoch": [], "train_loss": []}
for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    loss_sum, n_batch = 0.0, 0
    t0 = time.time()
    for batch in train_loader:
        u = batch["user_idx"].to(DEVICE)
        i = batch["item_idx"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        un = batch["user_num"].to(DEVICE)
        uc = batch["user_cuisine"].to(DEVICE)
        inum = batch["item_num"].to(DEVICE)
        ic = batch["item_cat"].to(DEVICE)
        pred = model(u, i, un, uc, inum, ic)
        loss = bce(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        n_batch += 1
    train_loss = loss_sum / n_batch
    elapsed = time.time() - t0
    print(f"epoch {epoch:02d} | {elapsed:.0f}s | loss={train_loss:.4f}")
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)

torch.save(model.state_dict(), MODELS_DIR / "two_tower.pt")

**Pre-compute the item index.** This is the move that makes Two-Tower fast at serve time: encode all 9K items through the item tower once, save the result as a numpy matrix, and at request time it's just one numpy matmul to score everything. Real production with 100K+ items would put this matrix into FAISS, but at 9K brute-force numpy is already sub-millisecond.

In [ ]:
@torch.no_grad()
def precompute_item_index(model, n_items, dataset, device, batch_size=4096):
    model.eval()
    all_idx = np.arange(n_items, dtype=np.int64)
    out = np.zeros((n_items, model.user_tower.mlp[-1].out_features), dtype=np.float32)
    for start in range(0, n_items, batch_size):
        end = min(start + batch_size, n_items)
        idx_t = torch.from_numpy(all_idx[start:end]).to(device)
        item_num_t = dataset.item_num_t[start:end].to(device)
        item_cat_t = dataset.item_cat_t[start:end].to(device)
        emb = model.item_tower(idx_t, item_num_t, item_cat_t).cpu().numpy()
        out[start:end] = emb
    return out

t0 = time.time()
item_index = precompute_item_index(model, n_items, train_ds, DEVICE)
print(f"item index: {item_index.shape} in {time.time()-t0:.2f}s")
np.save(MODELS_DIR / "two_tower_item_index.npy", item_index)

**Evaluate Recall@K.** For each val positive, encode the user, dot against the full item index, take top-K, check whether the actual positive is in there. Recall@200 ≥ 90% is the bar — at lower recall, too many true positives drop out of the candidate set before DeepFM sees them, and the latency win comes at the cost of accuracy.

In [ ]:
@torch.no_grad()
def eval_recall_at_k(model, val_pos, user_encoder, item_encoder, dataset, item_index, ks=(10, 50, 100, 200, 500), device="cpu"):
    model.eval()
    user_ids = val_pos["user_id"].values
    biz_ids = val_pos["business_id"].values

    user_idx_arr = np.array([user_encoder.encode(u) for u in user_ids], dtype=np.int64)
    pos_item_idx_arr = np.array([item_encoder.encode(b) for b in biz_ids], dtype=np.int64)

    user_idx_t = torch.from_numpy(user_idx_arr).to(device)
    user_num_t = dataset.user_num_t[user_idx_t.cpu()].to(device)
    user_cuisine_t = dataset.user_cuisine_t[user_idx_t.cpu()].to(device)

    user_emb_all = []
    bs = 4096
    for s in range(0, len(user_idx_arr), bs):
        e = min(s + bs, len(user_idx_arr))
        u_emb = model.user_tower(user_idx_t[s:e], user_num_t[s:e], user_cuisine_t[s:e]).cpu().numpy()
        user_emb_all.append(u_emb)
    user_emb = np.vstack(user_emb_all)

    scores = user_emb @ item_index.T

    metrics = {}
    for k in ks:
        top_k_idx = np.argpartition(-scores, k, axis=1)[:, :k]
        hits = np.any(top_k_idx == pos_item_idx_arr[:, None], axis=1)
        metrics[f"Recall@{k}"] = float(hits.mean())
    return metrics, scores

val_pos_filtered = val_df[val_df["stars"] >= 4].copy()
val_pos_filtered = val_pos_filtered[val_pos_filtered["user_id"].isin(user_encoder.id_to_idx)]
val_pos_filtered = val_pos_filtered[val_pos_filtered["business_id"].isin(item_encoder.id_to_idx)]
print(f"val positives for recall eval: {len(val_pos_filtered)}")

metrics, scores = eval_recall_at_k(model, val_pos_filtered, user_encoder, item_encoder, train_ds, item_index, device=DEVICE)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

with open(MODELS_DIR / "two_tower_metrics.json", "w") as f:
    json.dump({"config": CONFIG, "history": history, "recall": metrics}, f, indent=2)

**Latency benchmark.** Compare the per-request scoring cost of the Two-Tower retrieval path against scoring all 9K items with DeepFM. The retrieval path is one user-tower forward (small MLP, batch=1) + one numpy matmul against the precomputed (9K, 32) index. The full-DeepFM path is 9K independent forward passes through the full ranker.

In [ ]:
@torch.no_grad()
def benchmark_two_tower_query(model, item_index, dataset, device, n_trials=100):
    model.eval()
    times = []
    for _ in range(n_trials):
        uid = np.random.randint(0, n_users)
        u_t = torch.tensor([uid], dtype=torch.int64, device=device)
        un_t = dataset.user_num_t[uid:uid+1].to(device)
        uc_t = dataset.user_cuisine_t[uid:uid+1].to(device)
        torch.mps.synchronize() if device == "mps" else None
        t0 = time.time()
        u_emb = model.user_tower(u_t, un_t, uc_t).cpu().numpy()
        scores = u_emb @ item_index.T
        top200 = np.argpartition(-scores[0], 200)[:200]
        torch.mps.synchronize() if device == "mps" else None
        times.append((time.time() - t0) * 1000)
    return np.array(times)

times = benchmark_two_tower_query(model, item_index, train_ds, DEVICE, n_trials=100)
print(f"Two-Tower retrieval latency over 100 trials:")
print(f"  median: {np.median(times):.2f} ms")
print(f"  p95:    {np.percentile(times, 95):.2f} ms")
print(f"  p99:    {np.percentile(times, 99):.2f} ms")

If Recall@200 holds at 90%+ and median latency stays in the tens of milliseconds, the S1 push path is feasible: the ranker pipeline becomes Two-Tower retrieval (top-200, ~50ms) → DeepFM rerank (200 items, ~10ms) → return top-K to the UI. The 9,022-restaurant brute-force scoring with DeepFM that the demo is using today would balloon to seconds at production scale, which Two-Tower fixes.